# ROME Submission Notebook
**Course:** LLM4ChipDesign | **Instructors:** Ramesh Karri, Weihua Xiao

This notebook contains:
- **Part I:** Mux hierarchy demo (mux2to1 → mux4to1 → mux8to1)
- **Part II:** Ripple-carry adder hierarchy (half_adder → full_adder → adder4 → adder8)
- **Part III:** Debugging documentation

# Initial Setup

In [1]:
#@title Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update -qq
!apt-get install -y iverilog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 4.8 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  gtkwave
The following NEW packages will be installed:
  iverilog
0 upgraded, 1 newly installed, 0 to remove and 104 not upgraded.
Need to get 2,130 kB of archives.
After this operation, 6,749 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 iverilog amd64 11.0-1.1 [2,130 kB]
Fetched 2,130 kB in 1s (2,849 kB/s)
Selecting previously unselected package iverilog.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../iverilog_11.0-1.1_amd64.deb ...
Unpacking iverilog (11.0-1.1) ...
Setting up iverilog (11.0-1.1) ...

In [2]:
#@title Select Model
# Uncomment EXACTLY ONE of the lines below to choose your LLM:
model_choice = "gpt-4o"                    # ChatGPT (OpenAI)
#model_choice = "claude-sonnet-4-5"        # Claude (Anthropic)
#model_choice = "gemini-2.5-flash"         # Gemini (Google)
print(f"Model selected: {model_choice}")

Model selected: gpt-4o


In [3]:
#@title Utility functions

import sys
import os
import re
import time
import subprocess
import numpy as np
from abc import ABC, abstractmethod

try:
    import openai
except ImportError:
    openai = None

try:
    import anthropic
except ImportError:
    anthropic = None

try:
    from google import genai
    from google.genai import types
except ImportError:
    genai = None
    types = None


################################################################################
### LOGGING
################################################################################
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
################################################################################
class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def get_last_n_messages(self, n):
        return self.messages[-n:]

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        self.messages = []

    def __str__(self):
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
################################################################################
class AbstractLLM(ABC):
    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        pass


class ChatGPT(AbstractLLM):
    def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        # Preserve roles: 'system', 'user', 'assistant' are all valid for OpenAI
        messages = [{"role": msg["role"], "content": msg["content"]}
                    for msg in conversation.get_messages()]
        base_delay = 2
        max_retries = 10
        for attempt in range(1, max_retries + 1):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_id,
                    messages=messages,
                )
                return response.choices[0].message.content
            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] OpenAI API error: {e}. Retrying in {wait_time:.1f}s...")
                time.sleep(wait_time)
        print("Failed, exceeded max retries")
        return 0


class Claude(AbstractLLM):
    def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ['CLAUDE_API_KEY'])
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20
        for attempt in range(1, max_retries + 1):
            try:
                output = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=16384,
                    messages=[{"role": msg["role"], "content": msg["content"]}
                               for msg in conversation.get_messages()]
                ).content[0].text
                return output
            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f}s...")
                time.sleep(wait_time)
        print(f"Failed, exceeded max retries {max_retries}")
        return 0


class Gemini(AbstractLLM):
    def __init__(self, model_id=model_choice):
        super().__init__()
        self.gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        output = self.gemini_client.models.generate_content(
            model=self.model_id,
            contents=[msg["content"] for msg in conversation.get_messages()],
            config=types.GenerateContentConfig(
                max_output_tokens=16384,
                temperature=0.6,
                topP=0.95,
            )
        ).text
        return output


################################################################################
### PARSING AND TEXT MANIPULATION
################################################################################
def find_verilog_modules(markdown_string, module_name='top_module'):
    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'
    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'
    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)
    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)
    module_matches = module_matches1 + module_matches2
    if not module_matches:
        return []
    return module_matches


def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)
    if not code_match:
        raise ValueError(f"No Verilog module blocks found in LLM response for '{module_name}'. "
                         f"Response was:\n{markdown_string[:500]}")
    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')


def generate_verilog(conv, model_type, model_id=""):
    if model_type == "ChatGPT":
        model = ChatGPT()
    elif model_type == "Claude":
        model = Claude()
    elif model_type == "Gemini":
        model = Gemini()
    else:
        raise ValueError("Invalid model type")
    return model.generate(conv)


print("Utility functions loaded successfully.")

Utility functions loaded successfully.


In [4]:
#@title Feedback Loop
def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, outdir="", log=None, prev_module=None):

    # Normalize outdir — always ends with /
    if outdir:
        outdir = outdir.rstrip('/') + '/'

    conv = Conversation(log_file=log)

    system_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is either incorrect or has a compilation error, you will suggest corrections. "
        "You will not refuse. You will not generate explanations, only code. "
        "Format your response as Verilog code containing the complete corrected module. Do not generate test benches."
    )

    # ChatGPT supports a true 'system' role; Claude requires 'user' for the first message
    if model_type == "ChatGPT":
        conv.add_message("system", system_prompt)
    elif model_type == "Claude":
        conv.add_message("user", system_prompt)
    # Gemini ignores a system seed; just add the design prompt directly

    print(f"\n{'='*60}")
    print(f"Generating module: {module}")
    print(f"Model: {model_type}")
    print(f"Testbench: {testbench}")
    print(f"Prompt preview (first 300 chars):\n{design_prompt[:300]}...")
    print('='*60)

    conv.add_message("user", design_prompt)

    success = False
    timeout = False
    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []
    filename = os.path.join(outdir, module + ".v")
    status = ""

    while not (success or timeout):
        start_total = time.time()
        print(f"\n[Iteration {iterations}] Calling {model_type} API...")
        response = generate_verilog(conv, model_type)
        if response == 0:
            print("LLM returned no response — aborting.")
            break
        end_gen = time.time()
        print(f"[Iteration {iterations}] LLM responded ({len(response)} chars). Writing to {filename}...")
        start_error = time.time()

        if prev_module is None:
            conv.add_message("assistant", response)
        else:
            with open(prev_module, "r") as f:
                prevmodule = f.read()
            response = prevmodule + "\n" + response
            conv.add_message("assistant", response)

        try:
            write_code_blocks_to_file(response, module, filename)
        except ValueError as e:
            print(f"[Iteration {iterations}] WARNING: {e}")
            status = "No Verilog found in response"
            message = f"Your response did not contain a valid Verilog module. Please respond with ONLY synthesizable Verilog code for module {module}. No explanations, no markdown, just the module code."
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", message)
            if iterations >= max_iterations:
                timeout = True
            iterations += 1
            continue

        print(f"[Iteration {iterations}] Running iverilog compile...")
        proc = subprocess.run(
            ["iverilog", "-o", os.path.join(outdir, module), filename, testbench],
            capture_output=True, text=True
        )

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(f"[Iteration {iterations}] {status}")
            print(f"iverilog stderr:\n{proc.stderr}")
            message = "The testbench failed to compile. Please fix the module. iverilog output:\n" + proc.stderr
        elif proc.stderr != "":
            status = "Warnings compiling testbench"
            print(f"[Iteration {iterations}] {status}")
            print(f"iverilog warnings:\n{proc.stderr}")
            message = "The testbench compiled with warnings. Please fix the module. iverilog output:\n" + proc.stderr
        else:
            print(f"[Iteration {iterations}] Compile OK. Running simulation with vvp...")
            proc = subprocess.run(
                ["vvp", os.path.join(outdir, module)],
                capture_output=True, text=True
            )
            print(f"Simulation output:\n{proc.stdout}")
            lines = proc.stdout.strip().split('\n')
            passed = any('passed!' in line for line in lines)
            if not passed:
                status = "Error running testbench"
                print(f"[Iteration {iterations}] {status}")
                message = "The testbench simulated but had errors. Please fix the module. vvp output:\n" + proc.stdout
            else:
                status = "Testbench ran successfully"
                print(f"[Iteration {iterations}] ✓ {status}")
                message = ""
                success = True

        with open(os.path.join(outdir, "log_iter_" + str(iterations) + ".txt"), 'w') as file:
            file.write('\n'.join(str(i) for i in conv.get_messages()))
            file.write('\n\n Iteration status: ' + status + '\n')

        if not success:
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print(f"\nFinished {module}: {'SUCCESS' if success else 'FAILED/TIMEOUT'}")
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))
    return (np.sum(timelist_total), np.sum(timelist_gen), np.sum(timelist_error))

print("Feedback loop loaded.")

Feedback loop loaded.


In [5]:
#@title Hierarchical Loop
def hier_gen(submods, max_iterations=10):
    totaltime = []
    gentime = []
    errortime = []
    done = ""
    for i in range(len(submods)):
        curr = submods[i][1]
        fcurr = submods[i][0]
        iocurr = submods[i][2]
        overall = submods[-1][1]

        if not os.path.isdir(fcurr):
            os.mkdir(fcurr)

        if i == 0:
            # First module — no prior context
            prompt = (
                "//We will be generating a " + overall + " hierarchically in Verilog. "
                "Please begin by generating a " + curr + " defined as follows:\n"
                "module " + fcurr + "(" + iocurr + ")\n"
                "//Insert code here\nendmodule"
            )
        else:
            # Middle and final modules — include ALL previously generated submodules
            all_prev_code = ""
            for j in range(i):
                fprev = submods[j][0]
                filecurr = "./" + fprev + "/" + fprev + ".v"
                with open(filecurr, "r") as f:
                    all_prev_code += f.read() + "\n"
            prompt = (
                "//We are generating a " + overall + " hierarchically in Verilog. "
                "We have already generated the following modules:\n"
                + all_prev_code +
                "\n//Please include all previous module(s) verbatim in your response and use them to "
                "hierarchically generate a " + curr + " defined as:\n"
                "module " + fcurr + "(" + iocurr + ")\n"
                "//Insert code here\nendmodule"
            )

        module = fcurr
        testbench = "./" + fcurr + "tb.v"
        model = os.environ["MODEL"]
        outdir = "./" + fcurr
        log = "./" + fcurr + "/log.txt"

        total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
        totaltime.append(total)
        gentime.append(gen)
        errortime.append(error)
        done = done + curr + ", "

    print("Overall Total time: ", np.sum(totaltime))
    print("Overall Generation Time: ", np.sum(gentime))
    print("Overall Error handling time: ", np.sum(errortime))

print("Hierarchical loop loaded.")

Hierarchical loop loaded.


# Setting the API Key
> **Important:** Insert your API key below. Never commit keys to a public repo.

In [6]:
# ---- INSERT YOUR API KEY HERE ----
# Uncomment and fill in the key matching your chosen model_choice above.

# For ChatGPT (gpt-4o):
os.environ['OPENAI_API_KEY'] = " "   # <-- replace with your key
os.environ["MODEL"] = "ChatGPT"

# For Claude:
# os.environ['CLAUDE_API_KEY'] = "YOUR_CLAUDE_API_KEY_HERE"
# os.environ["MODEL"] = "Claude"

# For Gemini:
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY_HERE"
# os.environ["MODEL"] = "Gemini"

print(f"Model set to: {os.environ['MODEL']}")
print("API key loaded (value hidden).")

Model set to: ChatGPT
API key loaded (value hidden).


---
# Part I — Mux Hierarchy Demo
Generates `mux2to1`, `mux4to1`, and `mux8to1` using the hierarchical loop, each verified with a self-checking testbench.

In [7]:
#@title Write Mux Testbenches
import os

# ---- mux2to1 testbench ----
mux2to1_tb = """
`timescale 1ns/1ps
module mux2to1tb;
    reg in1, in2, select;
    wire out;
    integer errors = 0;

    mux2to1 uut (.in1(in1), .in2(in2), .select(select), .out(out));

    task check;
        input exp;
        #1;
        if (out !== exp) begin
            $display("FAIL: in1=%b in2=%b sel=%b | got %b expected %b", in1, in2, select, out, exp);
            errors = errors + 1;
        end
    endtask

    initial begin
        // select=0 → out = in1
        in1=0; in2=0; select=0; check(0);
        in1=1; in2=0; select=0; check(1);
        in1=0; in2=1; select=0; check(0);
        in1=1; in2=1; select=0; check(1);
        // select=1 → out = in2
        in1=0; in2=0; select=1; check(0);
        in1=1; in2=0; select=1; check(0);
        in1=0; in2=1; select=1; check(1);
        in1=1; in2=1; select=1; check(1);

        if (errors == 0)
            $display("All tests passed!");
        else
            $display("%0d test(s) failed.", errors);
        $finish;
    end
endmodule
"""

# ---- mux4to1 testbench ----
mux4to1_tb = """
`timescale 1ns/1ps
module mux4to1tb;
    reg [1:0] sel;
    reg [3:0] in;
    wire out;
    integer errors = 0;

    mux4to1 uut (.sel(sel), .in(in), .out(out));

    integer i;
    reg expected;

    initial begin
        for (i = 0; i < 64; i = i + 1) begin
            {sel, in} = i[5:0];
            #2;
            expected = in[sel];
            if (out !== expected) begin
                $display("FAIL: sel=%b in=%b | got %b expected %b", sel, in, out, expected);
                errors = errors + 1;
            end
        end
        if (errors == 0)
            $display("All tests passed!");
        else
            $display("%0d test(s) failed.", errors);
        $finish;
    end
endmodule
"""

# ---- mux8to1 testbench ----
mux8to1_tb = """
`timescale 1ns/1ps
module mux8to1tb;
    reg [2:0] sel;
    reg [7:0] in;
    wire out;
    integer errors = 0;

    mux8to1 uut (.sel(sel), .in(in), .out(out));

    integer i;
    reg expected;

    initial begin
        for (i = 0; i < 2048; i = i + 1) begin
            {sel, in} = i[10:0];
            #2;
            expected = in[sel];
            if (out !== expected) begin
                $display("FAIL: sel=%b in=%b | got %b expected %b", sel, in, out, expected);
                errors = errors + 1;
            end
        end
        if (errors == 0)
            $display("All tests passed!");
        else
            $display("%0d test(s) failed.", errors);
        $finish;
    end
endmodule
"""

# Write testbench files (the hier_gen function looks for ./mux2to1tb.v etc.)
with open("mux2to1tb.v", "w") as f:
    f.write(mux2to1_tb)
with open("mux4to1tb.v", "w") as f:
    f.write(mux4to1_tb)
with open("mux8to1tb.v", "w") as f:
    f.write(mux8to1_tb)

print("Mux testbench files written:")
print("  mux2to1tb.v")
print("  mux4to1tb.v")
print("  mux8to1tb.v")

Mux testbench files written:
  mux2to1tb.v
  mux4to1tb.v
  mux8to1tb.v


In [8]:
#@title Define Mux Submodules
submodules_mux = [
    ["mux2to1", "2-to-1 multiplexer",  "input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4to1", "4-to-1 multiplexer",  "input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8to1", "8-to-1 multiplexer",  "input [2:0] sel, input [7:0] in, output reg out"],
]

In [9]:
#@title Run Mux Hierarchy Generation
hier_gen(submodules_mux)


Generating module: mux2to1
Model: ChatGPT
Testbench: ./mux2to1tb.v
Prompt preview (first 300 chars):
//We will be generating a 8-to-1 multiplexer hierarchically in Verilog. Please begin by generating a 2-to-1 multiplexer defined as follows:
module mux2to1(input wire in1, input wire in2, input wire select, output wire out)
//Insert code here
endmodule...

[Iteration 0] Calling ChatGPT API...
[Iteration 0] LLM responded (147 chars). Writing to ./mux2to1/mux2to1.v...
[Iteration 0] Running iverilog compile...
[Iteration 0] Error compiling testbench
iverilog stderr:
./mux2to1tb.v:10: error: Task body with multiple statements requires SystemVerilog.
./mux2to1tb.v:1: error: Task body with multiple statements requires SystemVerilog.


[Iteration 1] Calling ChatGPT API...
[Iteration 1] LLM responded (483 chars). Writing to ./mux2to1/mux2to1.v...
[Iteration 1] Running iverilog compile...
[Iteration 1] Error compiling testbench
iverilog stderr:
./mux2to1tb.v:10: error: Task body with multiple st

---
# Part II — Ripple-Carry Adder Hierarchy
Extension design: `half_adder` → `full_adder` → `adder4` → `adder8`

**Decomposition:**
- `half_adder`: adds two 1-bit inputs, produces sum and carry
- `full_adder`: uses two `half_adder` instances + OR gate for carry-in support
- `adder4`: 4-bit ripple-carry adder built from four `full_adder` instances
- `adder8`: 8-bit ripple-carry adder built from two `adder4` instances

In [10]:
#@title Write Adder Testbenches

# ---- half_adder testbench ----
half_adder_tb = """
`timescale 1ns/1ps
module half_addertb;
    reg a, b;
    wire sum, carry;
    integer errors = 0;

    half_adder uut (.a(a), .b(b), .sum(sum), .carry(carry));

    task check;
        input exp_sum, exp_carry;
        #1;
        if (sum !== exp_sum || carry !== exp_carry) begin
            $display("FAIL: a=%b b=%b | got sum=%b carry=%b, expected sum=%b carry=%b",
                     a, b, sum, carry, exp_sum, exp_carry);
            errors = errors + 1;
        end
    endtask

    initial begin
        a=0; b=0; check(0, 0);
        a=0; b=1; check(1, 0);
        a=1; b=0; check(1, 0);
        a=1; b=1; check(0, 1);
        if (errors == 0)
            $display("All tests passed!");
        else
            $display("%0d test(s) failed.", errors);
        $finish;
    end
endmodule
"""

# ---- full_adder testbench ----
full_adder_tb = """
`timescale 1ns/1ps
module full_addertb;
    reg a, b, cin;
    wire sum, cout;
    integer errors = 0;

    full_adder uut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer i;
    reg exp_sum, exp_cout;

    initial begin
        for (i = 0; i < 8; i = i + 1) begin
            {cin, b, a} = i[2:0];
            #2;
            {exp_cout, exp_sum} = a + b + cin;
            if (sum !== exp_sum || cout !== exp_cout) begin
                $display("FAIL: a=%b b=%b cin=%b | got sum=%b cout=%b, expected sum=%b cout=%b",
                         a, b, cin, sum, cout, exp_sum, exp_cout);
                errors = errors + 1;
            end
        end
        if (errors == 0)
            $display("All tests passed!");
        else
            $display("%0d test(s) failed.", errors);
        $finish;
    end
endmodule
"""

# ---- adder4 testbench ----
adder4_tb = """
`timescale 1ns/1ps
module adder4tb;
    reg [3:0] a, b;
    reg cin;
    wire [3:0] sum;
    wire cout;
    integer errors = 0;

    adder4 uut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer i, j;
    reg [4:0] expected;

    initial begin
        // Exhaustive test for a few representative values
        for (i = 0; i < 16; i = i + 1) begin
            for (j = 0; j < 16; j = j + 1) begin
                a = i; b = j; cin = 0; #2;
                expected = a + b + cin;
                if ({cout,sum} !== expected) begin
                    $display("FAIL: a=%b b=%b cin=%b | got cout=%b sum=%b, expected %b",
                             a, b, cin, cout, sum, expected);
                    errors = errors + 1;
                end
                a = i; b = j; cin = 1; #2;
                expected = a + b + cin;
                if ({cout,sum} !== expected) begin
                    $display("FAIL: a=%b b=%b cin=%b | got cout=%b sum=%b, expected %b",
                             a, b, cin, cout, sum, expected);
                    errors = errors + 1;
                end
            end
        end
        if (errors == 0)
            $display("All tests passed!");
        else
            $display("%0d test(s) failed.", errors);
        $finish;
    end
endmodule
"""

# ---- adder8 testbench ----
adder8_tb = """
`timescale 1ns/1ps
module adder8tb;
    reg [7:0] a, b;
    reg cin;
    wire [7:0] sum;
    wire cout;
    integer errors = 0;

    adder8 uut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer i;
    reg [8:0] expected;

    initial begin
        // Test 200 random combinations
        for (i = 0; i < 200; i = i + 1) begin
            a = $random; b = $random; cin = $random;
            #2;
            expected = a + b + cin;
            if ({cout, sum} !== expected) begin
                $display("FAIL: a=%b b=%b cin=%b | got cout=%b sum=%b, expected %b",
                         a, b, cin, cout, sum, expected);
                errors = errors + 1;
            end
        end
        // Corner cases
        a=8'hFF; b=8'hFF; cin=1; #2;
        expected = a + b + cin;
        if ({cout,sum} !== expected) begin
            $display("FAIL corner: a=FF b=FF cin=1"); errors = errors + 1;
        end
        a=8'h00; b=8'h00; cin=0; #2;
        expected = 0;
        if ({cout,sum} !== expected) begin
            $display("FAIL corner: a=00 b=00 cin=0"); errors = errors + 1;
        end

        if (errors == 0)
            $display("All tests passed!");
        else
            $display("%0d test(s) failed.", errors);
        $finish;
    end
endmodule
"""

with open("half_addertb.v", "w") as f:
    f.write(half_adder_tb)
with open("full_addertb.v", "w") as f:
    f.write(full_adder_tb)
with open("adder4tb.v", "w") as f:
    f.write(adder4_tb)
with open("adder8tb.v", "w") as f:
    f.write(adder8_tb)

print("Adder testbench files written:")
for tb in ["half_addertb.v", "full_addertb.v", "adder4tb.v", "adder8tb.v"]:
    print(f"  {tb}")

Adder testbench files written:
  half_addertb.v
  full_addertb.v
  adder4tb.v
  adder8tb.v


In [11]:
#@title Define Adder Submodules
submodules_adder = [
    ["half_adder", "half adder",
     "input a, input b, output sum, output carry"],

    ["full_adder", "full adder (instantiating two half_adder modules)",
     "input a, input b, input cin, output sum, output cout"],

    ["adder4", "4-bit ripple-carry adder (instantiating four full_adder modules)",
     "input [3:0] a, input [3:0] b, input cin, output [3:0] sum, output cout"],

    ["adder8", "8-bit ripple-carry adder (instantiating two adder4 modules)",
     "input [7:0] a, input [7:0] b, input cin, output [7:0] sum, output cout"],
]

In [12]:
#@title Run Adder Hierarchy Generation
hier_gen(submodules_adder)


Generating module: half_adder
Model: ChatGPT
Testbench: ./half_addertb.v
Prompt preview (first 300 chars):
//We will be generating a 8-bit ripple-carry adder (instantiating two adder4 modules) hierarchically in Verilog. Please begin by generating a half adder defined as follows:
module half_adder(input a, input b, output sum, output carry)
//Insert code here
endmodule...

[Iteration 0] Calling ChatGPT API...
[Iteration 0] LLM responded (133 chars). Writing to ./half_adder/half_adder.v...
[Iteration 0] Running iverilog compile...
[Iteration 0] Error compiling testbench
iverilog stderr:
./half_addertb.v:10: error: Task body with multiple statements requires SystemVerilog.
./half_addertb.v:1: error: Task body with multiple statements requires SystemVerilog.


[Iteration 1] Calling ChatGPT API...
[Iteration 1] LLM responded (133 chars). Writing to ./half_adder/half_adder.v...
[Iteration 1] Running iverilog compile...
[Iteration 1] Error compiling testbench
iverilog stderr:
./half_addertb.

---
# Part III — Debugging Loop Documentation

The cell below reads and displays the iteration logs generated by `hier_gen` for the adder hierarchy. This shows the iterative debugging process automatically.

You will also fill in the summary table in `ROME_Report.pdf` based on what you observe here.

In [13]:
#@title Display Iteration Logs for Adder Hierarchy

import glob

adder_modules = ["half_adder", "full_adder", "adder4", "adder8"]

for mod in adder_modules:
    log_files = sorted(glob.glob(f"./{mod}/log_iter_*.txt"))
    if not log_files:
        print(f"\n{'='*60}")
        print(f"Module: {mod} — No iteration logs found (may not have been run yet)")
        continue
    print(f"\n{'='*60}")
    print(f"Module: {mod} — {len(log_files)} iteration(s)")
    print('='*60)
    for log_path in log_files:
        iter_num = log_path.split("log_iter_")[1].replace(".txt", "")
        with open(log_path, "r") as f:
            content = f.read()
        # Extract just the status line for brevity
        status_line = [l for l in content.split("\n") if "Iteration status" in l]
        status = status_line[0] if status_line else "(status not found)"
        print(f"\n--- Iteration {iter_num} ---")
        print(status)
        # Print last 30 lines of the log for context
        lines = content.strip().split("\n")
        print("\n".join(lines[-30:]))


Module: half_adder — 11 iteration(s)

--- Iteration 0 ---
 Iteration status: Error compiling testbench
{'role': 'system', 'content': 'You are an autocomplete engine for Verilog code. Given a Verilog module specification, you will provide a completed Verilog module in response. You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. Given a Verilog module that is either incorrect or has a compilation error, you will suggest corrections. You will not refuse. You will not generate explanations, only code. Format your response as Verilog code containing the complete corrected module. Do not generate test benches.'}
{'role': 'user', 'content': '//We will be generating a 8-bit ripple-carry adder (instantiating two adder4 modules) hierarchically in Verilog. Please begin by generating a half adder defined as follows:\nmodule half_adder(input a, input b, output sum, output carry)\n//Insert code here\nendmodule'}
{'role': 'assistant', 'c

In [14]:
#@title Show Final Generated Verilog Files

print("\n=== PART I: MUX HIERARCHY ===")
for mod in ["mux2to1", "mux4to1", "mux8to1"]:
    vfile = f"./{mod}/{mod}.v"
    if os.path.exists(vfile):
        print(f"\n--- {vfile} ---")
        with open(vfile) as f:
            print(f.read())
    else:
        print(f"\n{vfile} not found")

print("\n=== PART II: ADDER HIERARCHY ===")
for mod in ["half_adder", "full_adder", "adder4", "adder8"]:
    vfile = f"./{mod}/{mod}.v"
    if os.path.exists(vfile):
        print(f"\n--- {vfile} ---")
        with open(vfile) as f:
            print(f.read())
    else:
        print(f"\n{vfile} not found")


=== PART I: MUX HIERARCHY ===

--- ./mux2to1/mux2to1.v ---
module mux2to1(input wire in1, input wire in2, input wire select, output wire out);
    assign out = select ? in2 : in1;
endmodule


--- ./mux4to1/mux4to1.v ---
module mux2to1(input wire in1, input wire in2, input wire select, output wire out);
    assign out = select ? in2 : in1;
endmodule
module mux4to1(input [1:0] sel, input [3:0] in, output wire out);
    wire out1, out2;
    
    mux2to1 mux0(.in1(in[0]), .in2(in[1]), .select(sel[0]), .out(out1));
    mux2to1 mux1(.in1(in[2]), .in2(in[3]), .select(sel[0]), .out(out2));
    mux2to1 mux2(.in1(out1), .in2(out2), .select(sel[1]), .out(out));
endmodule


--- ./mux8to1/mux8to1.v ---
module mux2to1(input wire in1, input wire in2, input wire select, output wire out);
    assign out = select ? in2 : in1;
endmodule
module mux4to1(input [1:0] sel, input [3:0] in, output wire out);
    wire out1, out2;
    
    mux2to1 mux0(.in1(in[0]), .in2(in[1]), .select(sel[0]), .out(out1));
    